In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

required_settings = {
    "GROQ_API_KEY": GROQ_API_KEY,
    "NEO4J_URI": NEO4J_URI,
    "NEO4J_USERNAME": NEO4J_USERNAME,
    "NEO4J_PASSWORD": NEO4J_PASSWORD,
}
missing_settings = [name for name, value in required_settings.items() if not value]
if missing_settings:
    raise RuntimeError(f"Missing settings in .env: {', '.join(missing_settings)}")

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    api_key=GROQ_API_KEY
)

In [ ]:
from langchain_core.prompts import PromptTemplate

qa_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You answer questions using data retrieved from a graph database.

The user's question is:
{question}

The database returned these records:
{context}

Instructions:
- Read the returned records carefully.
- The records contain the answer if they include relevant values.
- For movie titles, return the movie title or titles.
- Never say "I don't know" when the records contain the answer.
- Only say "I don't know" if the records contain no relevant information.

Answer:
"""
)


cypher_prompt = PromptTemplate(
    input_variables=["schema", "question"],
    template="""
You generate Cypher queries for a Neo4j knowledge graph.

Graph schema (includes example stored values for properties):
{schema}

Rules:
- Use only labels, relationships, and properties present in the schema.
- Match entity and genre names to the stored values shown in the schema.
- Treat equivalent wording, hyphenation, and spacing as natural-language variations.
  For example, "science-fiction" and "science fiction" may refer to
  the same genre if the graph stores "Science Fiction".
- Compare names and titles case-insensitively, for example
  WHERE toLower(m.title) = toLower("dunkirk")
  instead of MATCH (m:Movie {{title: "dunkirk"}}).
- Follow the relationships in the schema to answer the complete question.
- Return only a valid, read-only Cypher query.
- Do not create, update, or delete graph data.

Question: {question}

Cypher query:
"""
)

In [ ]:
from typing import Optional

from pydantic import BaseModel, Field


class QuestionIntent(BaseModel):
    """How to route a user's movie question."""

    is_recommendation: bool = Field(
        description="True if the user wants movies similar to a movie they name."
    )
    movie_title: Optional[str] = Field(
        default=None,
        description="The movie the user likes, exactly as written, if one is named.",
    )


intent_llm = llm.with_structured_output(QuestionIntent)


def detect_intent(question: str) -> QuestionIntent:
    return intent_llm.invoke(
        "Decide whether this question asks for movie recommendations based on a "
        "movie the user likes, and extract that movie's title.\n\n"
        f"Question: {question}"
    )

In [ ]:
from langchain_neo4j import Neo4jGraph, GraphCypherQAChain


# Connect to Neo4j. enhanced_schema adds example property values (such as
# genre names) to the schema, so the LLM can match the values actually stored.
graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    enhanced_schema=True,
)

In [ ]:
chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,
    cypher_prompt=cypher_prompt,  # your existing prompt, if defined
    qa_prompt=qa_prompt,
    verbose=True,
    return_intermediate_steps=True,
    allow_dangerous_requests=True,
)


In [ ]:
FIND_MOVIE_QUERY = """
MATCH (m:Movie)
WHERE toLower(m.title) = toLower($title)
RETURN m.title AS title
LIMIT 1
"""

# Same scoring query as README section 6: shared actors (3 points each),
# shared directors (2 points each), and shared genres (1 point each).
RECOMMEND_QUERY = """
MATCH (favorite:Movie {title: $favoriteMovie})
MATCH (rec:Movie)
WHERE rec <> favorite

OPTIONAL MATCH
    (favorite)<-[:ACTED_IN]-(actor:Person)-[:ACTED_IN]->(rec)
WITH favorite, rec,
     count(DISTINCT actor) AS sharedActors

OPTIONAL MATCH
    (favorite)<-[:DIRECTED]-(director:Person)-[:DIRECTED]->(rec)
WITH favorite, rec, sharedActors,
     count(DISTINCT director) AS sharedDirectors

OPTIONAL MATCH
    (favorite)-[:IN_GENRE]->(genre:Genre)<-[:IN_GENRE]-(rec)
WITH rec, sharedActors, sharedDirectors,
     count(DISTINCT genre) AS sharedGenres

WITH rec, sharedActors, sharedDirectors, sharedGenres,
     sharedActors * 3 + sharedDirectors * 2 + sharedGenres AS score
WHERE score > 0

RETURN rec.title AS recommendation,
       sharedActors,
       sharedDirectors,
       sharedGenres,
       score
ORDER BY score DESC, recommendation
LIMIT 10
"""


def recommend(title: str) -> str:
    found = graph.query(FIND_MOVIE_QUERY, {"title": title.strip()})
    if not found:
        return f'I couldn\'t find "{title}" in the movie graph.'

    favorite = found[0]["title"]
    rows = graph.query(RECOMMEND_QUERY, {"favoriteMovie": favorite})
    if not rows:
        return f"I couldn't find any movies similar to {favorite}."

    lines = [f"Because you like {favorite}, you might enjoy:"]
    for row in rows:
        reasons = []
        if row["sharedActors"]:
            reasons.append(f"{row['sharedActors']} shared actor(s)")
        if row["sharedDirectors"]:
            reasons.append(f"{row['sharedDirectors']} shared director(s)")
        if row["sharedGenres"]:
            reasons.append(f"{row['sharedGenres']} shared genre(s)")
        lines.append(f"- {row['recommendation']} ({', '.join(reasons)})")
    return "\n".join(lines)


def ask(question: str) -> str:
    intent = detect_intent(question)
    if intent.is_recommendation and intent.movie_title:
        return recommend(intent.movie_title)
    return chain.invoke({"query": question})["result"]

In [ ]:
# Type a question when prompted, or replace input(...) with a string,
# for example: question = "I like Dunkirk, what else should I watch?"
question = input("Ask a question about movies: ").strip()

if not question:
    print("Please enter a question.")
else:
    try:
        print(ask(question))
    except Exception as error:
        print(f"Could not answer the question: {error}")